---


 The performance of a machine learning model can be characterized in terms of the bias and the variance of the model.


---

##### Bias, Variance, and Irreducible Error

Consider a machine learning model that makes predictions for a predictive modeling task, such as regression or classification.

The performance of the model on the task can be described in terms of the prediction error on all examples not used to train the model. We will refer to this as the model error.

Error(Model)

The model error can be decomposed into three sources of error: <b>the variance of the model, the bias of the model, and the variance of the irreducible error in the data<b>.


###### Error(Model) = Variance(Model) + Bias(Model) + Variance(Irreducible Error)


In [ ]:
! pip install mlxtend

### estimate the bias and variance for a regression model

In [1]:
from pandas import read_csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from mlxtend.evaluate import bias_variance_decomp

# load dataset
dataframe = read_csv('boston.csv')
dataframe.head()

,ID,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,black,lstat,medv
0,1,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,2,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,4,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
3,5,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2
4,7,0.08829,12.5,7.87,0,0.524,6.012,66.6,5.5605,5,311,15.2,395.60,12.43,22.9


In [3]:

# separate into inputs and outputs
data = dataframe.values
X, y = data[:, :-1], data[:, -1]


In [4]:
# split the data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=1)

In [5]:

# define the model
model = LinearRegression()

In [6]:
# estimate bias and variance
mse, bias, var = bias_variance_decomp(model, X_train, y_train, X_test, y_test, loss='mse', num_rounds=200, random_seed=1)


In [7]:
# summarize results
print('MSE: %.3f' % mse)
print('Bias: %.3f' % bias)
print('Variance: %.3f' % var)

MSE: 23.993
Bias: 21.834
Variance: 2.159


What happens internally?

    For num_rounds = 200:
    
    Repeatedly resample the training data (with replacement)
    
    Train a fresh Linear Regression each time
    
    Predict on the same test set
    
    Measure:
    
        How far predictions are from true values (bias)
        
        How much predictions change across runs (variance)

What these numbers mean mathematically

    Bias–variance decomposition (approximately):
    
    MSE ≈  Bias  + Variance

    Here:

        Bias ≈ 21.834 (dominant component)
        
        Variance ≈ 2.159 (very small)


Diagnosis: Bias vs Variance

    ✅ Bias is very HIGH

        Model has strong assumptions
        
        Linear Regression assumes pure linear relationship
        
        Boston housing data has non-linear, interacting features

    ➡️ Underfitting


✅ Variance is LOW

    Predictions don’t change much with different training samples
    
    Model is stable
    
    Not memorizing data

➡️ Not overfitting

High Bias + Low Variance → Underfitting

### DecisionTreeRegressor

In [ ]:
from mlxtend.evaluate import bias_variance_decomp
from sklearn.tree import DecisionTreeRegressor
from mlxtend.data import boston_housing_data
from sklearn.model_selection import train_test_split


X, y = boston_housing_data()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123, shuffle=True)

tree = DecisionTreeRegressor(random_state=123)

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
        tree, X_train, y_train, X_test, y_test, 
        loss='mse',
        random_seed=123)

print('Average expected loss: %.3f' % avg_expected_loss)
print('Average bias: %.3f' % avg_bias)
print('Average variance: %.3f' % avg_var)

✅ Bias = 14.096 → LOWER bias

Much lower than Linear Regression
(LR bias ≈ 21.834)

Tree captures non-linear patterns

➡️ Less underfitting

❌ Variance = 17.440 → VERY HIGH variance

Much higher than Linear Regression
(LR variance ≈ 2.159)

Model is highly sensitive to training data

➡️ Overfitting


Linear Regression showed high bias and low variance, indicating underfitting.

Decision Tree Regression showed reduced bias but significantly increased variance, indicating overfitting.

This confirms that optimal performance requires balancing bias and variance rather than minimizing either independently.


### Learning Curve case 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

data_file_path = 'diabetes.csv'
data_df = pd.read_csv(data_file_path)


In [ ]:
X = data_df.drop("Outcome", axis=1)
y = data_df["Outcome"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)



In [ ]:
# Store scores
train_score = []
test_score = []
k_vals = range(1, 21)

# Loop over different K values
for k in k_vals:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_score.append(knn.score(X_train, y_train))
    test_score.append(knn.score(X_test, y_test))

# Plot performance
plt.figure(figsize=(10,5))
plt.plot(k_vals, train_score, color='r', label='Training Score')
plt.plot(k_vals, test_score, color='b', label='Test Score')
plt.xlabel('K Value')
plt.ylabel('Accuracy')
plt.title('KNN Performance for Different K Values')
plt.legend()
plt.show()


In [ ]:
# Choose best K (example: 14)
knn = KNeighborsClassifier(n_neighbors=14)
knn.fit(X_train, y_train)
accuracy = knn.score(X_test, y_test)

print(f" Test Accuracy with K=14: {accuracy:.4f}")
